In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TopKSAE(nn.Module):
    """
    Top-K Sparse Autoencoder:
      h \in R^d  ->  z \in R^m (ReLU, non-negative) -> keep Top-K per sample  ->  \hat{h} \in R^d
    Encoder: Linear(d -> m) + bias
    Decoder: Linear(m -> d), no bias (standard in SAE work)
    """
    def __init__(self, d: int, m: int, k: int):
        super().__init__()
        self.d, self.m, self.k = d, m, k
        self.enc = nn.Linear(d, m, bias=True)
        self.dec = nn.Linear(m, d, bias=False)

        # Kaiming init for encoder; small-norm decoder helps early stability
        nn.init.kaiming_uniform_(self.enc.weight, a=math.sqrt(5))
        nn.init.uniform_(self.dec.weight, -1e-3, 1e-3)  

    @torch.no_grad()
    def _dead_code_resample(self, z_batch: torch.Tensor, min_fire: int = 10):
        """
        Simple dead-latent resampling: if a latent fired < min_fire times in the last epoch,
        re-init its decoder column and encoder row. See SAELens tutorials for more rigorous recipes.
        """
        fires = (z_batch > 0).sum(dim=0)  # [m]
        dead = torch.nonzero(fires < min_fire).flatten()
        if len(dead):
            self.dec.weight.data[dead].uniform_(-1e-3, 1e-3) #FIX - incorrect decoder resampling
            nn.init.kaiming_uniform_(self.enc.weight.data[dead], a=math.sqrt(5))

    def encode_logits(self, h: torch.Tensor) -> torch.Tensor:
        # h: [B, d] -> logits: [B, m]
        return self.enc(h)

    def encode(self, h: torch.Tensor) -> torch.Tensor:
        # non-negative codes (ReLU), then Top-K mask per sample
        logits = self.encode_logits(h)                      # [B, m]
        z = F.relu(logits)
        if self.k < self.m:
            topk_idx = torch.topk(z, self.k, dim=-1).indices
            mask = torch.zeros_like(z).scatter(1, topk_idx, 1.0)
            z = z * mask
        return z

    def forward(self, h: torch.Tensor):
        z = self.encode(h)                                 # [B, m]
        h_hat = self.dec(z)                                # [B, d]
        return h_hat, z
